## 02 — Plate-solving

Tests `extractor.platesolve` against a real FITS image in `../data/`.

Returns a `PlatesolveResult` with the updated header, all detected source
positions (`detected_x/y`), and the astrometry-confirmed subset (`matched_x/y`).

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import warnings

sys.path.insert(0, os.path.abspath('..'))

from extractor.platesolve import platesolve
from extractor.stars import extract_stars
from astropy.io import fits as afits
from astropy.wcs import WCS

In [ ]:
data_dir = Path('../data')
fits_files = sorted(data_dir.glob('*.fit')) + sorted(data_dir.glob('*.fits'))
if not fits_files:
    raise FileNotFoundError(f"No .fit / .fits files found in {data_dir.resolve()}")

image_path = fits_files[0]
print(f"Using: {image_path}")

with afits.open(image_path) as hdul:
    image = hdul[0].data.astype(float)

In [ ]:
# Preview star detection before submitting.
xs, ys, fluxes = extract_stars(image, max_sources=300)
print(f"Detected {len(xs)} sources")
if len(xs):
    print(f"Brightest flux : {fluxes[0]:.1f}")
    print(f"Faintest kept  : {fluxes[-1]:.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
lo, hi = np.percentile(image[np.isfinite(image)], [0.5, 99.5])
ax.imshow(np.arcsinh(np.clip(image, lo, hi)), origin='lower', cmap='gray',
          vmin=np.arcsinh(lo), vmax=np.arcsinh(hi))
ax.scatter(xs, ys, s=60, facecolors='none', edgecolors='lime',
           linewidths=0.8, label=f'{len(xs)} sources')
ax.legend(loc='upper right')
ax.set_title('Detected sources')
plt.tight_layout()
plt.show()

In [ ]:
# Plate-solve. write=False (default) — does NOT modify the file.
result = platesolve(
    image_path,
    use_source_list=True,
    max_sources=300,
    write=False,
    verbose=True,
)

In [ ]:
if result is None:
    print("Solve failed — check output above.")
else:
    print("Solve succeeded.")
    print(f"  RA   (CRVAL1) = {result.header['CRVAL1']:.6f} deg")
    print(f"  Dec  (CRVAL2) = {result.header['CRVAL2']:.6f} deg")
    print(f"  CTYPE1        = {result.header['CTYPE1']}")
    print(f"  Detected      = {len(result.detected_x)} sources")
    print(f"  Matched       = {len(result.matched_x)} sources")
    print()
    for k in ['CD1_1', 'CD1_2', 'CD2_1', 'CD2_2']:
        if k in result.header:
            print(f"  {k} = {result.header[k]:.6e}")

In [ ]:
# Plate scale and rotation from CD matrix.
if result is not None and 'CD1_1' in result.header:
    cd = np.array([[result.header['CD1_1'], result.header['CD1_2']],
                   [result.header['CD2_1'], result.header['CD2_2']]])
    scale_arcsec = np.mean(np.linalg.svd(cd, compute_uv=False)) * 3600.0

    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        wcs = WCS(result.header)
    cx = result.header.get('NAXIS1', image.shape[1]) / 2.0
    cy = result.header.get('NAXIS2', image.shape[0]) / 2.0
    pts = wcs.all_pix2world([[cx, cy], [cx, cy + 1.0]], 0)
    dra  = (pts[1, 0] - pts[0, 0]) * np.cos(np.radians(pts[0, 1]))
    ddec = pts[1, 1] - pts[0, 1]
    rot_deg = float(np.degrees(np.arctan2(-dra, ddec)))

    print(f"Plate scale : {scale_arcsec:.3f} arcsec/px")
    print(f"Rotation    : {rot_deg:.3f} deg (PA of detector +Y, east of north)")

In [ ]:
# WCS sky grid overlay.
if result is not None:
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        wcs = WCS(result.header)

    fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={'projection': wcs})
    lo, hi = np.percentile(image[np.isfinite(image)], [0.5, 99.5])
    ax.imshow(np.arcsinh(np.clip(image, lo, hi)), origin='lower', cmap='gray',
              vmin=np.arcsinh(lo), vmax=np.arcsinh(hi))
    ax.coords.grid(True, color='cyan', alpha=0.4, linestyle='--')
    ax.coords['ra'].set_axislabel('RA')
    ax.coords['dec'].set_axislabel('Dec')
    ax.set_title(f"{image_path.name} — plate-solved WCS overlay")
    plt.tight_layout()
    plt.show()